# 👗 Dress / Clothing Image Classifier
### Uses CLIP (Pre-trained by OpenAI) — NO TRAINING NEEDED
**Run each cell once. If session restarts, just re-run from Cell 1.**

In [1]:
# ── CELL 1: Install (run once) ─────────────────────────────────────────────
# This saves installed packages so session restart won't need full reinstall
import os

if not os.path.exists('/content/installed.flag'):
    print('Installing packages...')
    os.system('pip install -q transformers torch torchvision Pillow gradio')
    open('/content/installed.flag', 'w').close()
    print('Done!')
else:
    print('Already installed!')

Already installed!


In [2]:
# ── CELL 2: Load CLIP model (pre-trained, no training needed) ──────────────
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import gradio as gr
import numpy as np

print('Loading CLIP model (pre-trained by OpenAI)...')
model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model.eval()
print('Model ready!')

Loading CLIP model (pre-trained by OpenAI)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model ready!


In [3]:
!pip install -U gradio

In [4]:
import gradio as gr
print(gr.__version__)

6.15.1


In [ ]:
    # ── CELL 3: DressAI v5 — 20 Classes, Extra Charts, No Double Save ───────────


    import torch
    import numpy as np
    import gradio as gr
    from PIL import Image
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import io, csv, os, json, re

    # ── Load model if not already loaded (safe after session restart) ────────────
    try:
        processor
        model
        print("Model already loaded!")
    except NameError:
        print("Loading CLIP model...")
        from transformers import CLIPProcessor, CLIPModel
        model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        model.eval()
        print("Model ready!")

    from datetime import datetime
    from collections import Counter

    # ── 20 Classes ───────────────────────────────────────────────────────────────
    CLASS_DESCRIPTIONS = [
        "a casual dress or sundress or day dress",
        "a formal dress or evening gown or ball gown or cocktail dress",
        "a bodycon dress or tight fitted dress or figure hugging dress",
        "an ethnic wear like saree or kurti or salwar kameez or lehenga or churidar or anarkali",
        "a maxi dress or long flowy dress or floor length dress",
        "a shirt or formal shirt or casual shirt or button down shirt or oxford shirt",
        "trousers or pants or formal pants or chinos or slacks",
        "a traditional or cultural outfit or festival wear or regional dress",
        "a skirt or mini skirt or pleated skirt or pencil skirt or A-line skirt",
        "a jacket or blazer or suit jacket or sport coat",
        "a t-shirt or polo shirt or casual top or crop top or tank top",
        "a jumpsuit or romper or playsuit or dungaree or overalls",
        "jeans or denim jeans or skinny jeans or wide leg jeans or bootcut jeans",
        "a sweater or hoodie or sweatshirt or pullover or knitwear",
        "a coat or overcoat or trench coat or winter coat or puffer jacket",
        "shorts or bermuda shorts or denim shorts or casual shorts",
        "a swimwear or bikini or swimsuit or beachwear or one piece swimsuit",
        "a saree or Indian saree or silk saree or cotton saree",
        "a wedding dress or bridal gown or white wedding dress",
        "activewear or sportswear or gym wear or yoga pants or leggings or athletic wear"
    ]

    CLASS_LABELS = [
        'Casual Dress','Formal / Evening Dress','Bodycon Dress',
        'Ethnic Wear (Saree/Kurti/Lehenga)','Maxi Dress','Shirt',
        'Trousers / Formal Pants','Traditional / Festival Outfit',
        'Skirt','Jacket / Blazer','T-Shirt / Casual Top','Jumpsuit / Romper',
        'Jeans / Denim','Sweater / Hoodie / Knitwear','Coat / Overcoat / Puffer',
        'Shorts','Swimwear / Beachwear','Saree (Indian)',
        'Wedding / Bridal Dress','Activewear / Sportswear'
    ]

    CLASS_EMOJI = {
        'Casual Dress':'👗','Formal / Evening Dress':'🥂','Bodycon Dress':'✨',
        'Ethnic Wear (Saree/Kurti/Lehenga)':'🪷','Maxi Dress':'🌸','Shirt':'👔',
        'Trousers / Formal Pants':'👖','Traditional / Festival Outfit':'🎊',
        'Skirt':'👘','Jacket / Blazer':'🧥','T-Shirt / Casual Top':'👕',
        'Jumpsuit / Romper':'🤸','Jeans / Denim':'🔵',
        'Sweater / Hoodie / Knitwear':'🧶','Coat / Overcoat / Puffer':'🧤',
        'Shorts':'🩳','Swimwear / Beachwear':'🩱','Saree (Indian)':'🪷',
        'Wedding / Bridal Dress':'💍','Activewear / Sportswear':'💪'
    }

    STYLE_TIPS = {
        'Casual Dress':'Pair with sneakers or sandals for a relaxed daytime look.',
        'Formal / Evening Dress':'Add heels and a clutch for a polished, elegant appearance.',
        'Bodycon Dress':'Style with block heels and minimal jewelry to let the dress shine.',
        'Ethnic Wear (Saree/Kurti/Lehenga)':'Accessorize with traditional jewelry like jhumkas or bangles.',
        'Maxi Dress':'Add a denim jacket and wedges for a boho-chic vibe.',
        'Shirt':'Tuck into trousers or pair with jeans for a smart casual look.',
        'Trousers / Formal Pants':'Pair with a tucked-in blouse or a crisp formal shirt.',
        'Traditional / Festival Outfit':'Complete with traditional footwear and ethnic accessories.',
        'Skirt':'Pair with a fitted top or tucked-in blouse for balance.',
        'Jacket / Blazer':'Layer over a simple outfit to instantly elevate your look.',
        'T-Shirt / Casual Top':'Great with jeans, joggers, or layered under a jacket.',
        'Jumpsuit / Romper':'Add a belt to define the waist; heels or sneakers both work great.',
        'Jeans / Denim':'Extremely versatile — dress up with a blazer or down with a tee.',
        'Sweater / Hoodie / Knitwear':'Layer over a collared shirt or pair with slim-fit jeans.',
        'Coat / Overcoat / Puffer':'Wear over any outfit for warmth; belt a trench for a polished look.',
        'Shorts':'Pair with a tucked-in tee or a breezy blouse for summer outings.',
        'Swimwear / Beachwear':'Add a sarong or beach cover-up and sandals for a complete beach look.',
        'Saree (Indian)':'Pair with a matching blouse and heels or traditional footwear.',
        'Wedding / Bridal Dress':'Let the dress be the statement — keep accessories elegant and minimal.',
        'Activewear / Sportswear':'Choose moisture-wicking fabrics; pair with proper athletic footwear.'
    }

    OCCASIONS = {
        'Casual Dress':['Weekend Outing','Brunch','Shopping','Picnic'],
        'Formal / Evening Dress':['Wedding Reception','Office Party','Date Night','Award Ceremony'],
        'Bodycon Dress':['Night Out','Birthday Party','Club / Lounge','Cocktail Party'],
        'Ethnic Wear (Saree/Kurti/Lehenga)':['Festival','Wedding','Pooja / Temple','Family Function'],
        'Maxi Dress':['Beach Holiday','Garden Party','Vacation','Casual Outing'],
        'Shirt':['Office','Business Meeting','College','Casual Friday'],
        'Trousers / Formal Pants':['Office','College','Formal Event','Travel'],
        'Traditional / Festival Outfit':['Festival','Cultural Event','Temple Visit','Ceremony'],
        'Skirt':['College','Brunch','Shopping','Casual Party'],
        'Jacket / Blazer':['Office','Formal Event','Business Meeting','Travel'],
        'T-Shirt / Casual Top':['Gym','Home / Loungewear','Casual Day','Sports'],
        'Jumpsuit / Romper':['Vacation','Casual Party','Day Out','Brunch'],
        'Jeans / Denim':['Casual Day Out','College','Shopping','Travel'],
        'Sweater / Hoodie / Knitwear':['College','Casual Outing','Work from Home','Travel'],
        'Coat / Overcoat / Puffer':['Winter Outing','Office','Formal Event','Travel'],
        'Shorts':['Beach','Gym','Casual Day','Summer Outing'],
        'Swimwear / Beachwear':['Beach','Pool Party','Resort','Water Sports'],
        'Saree (Indian)':['Wedding','Festival','Pooja','Formal Function'],
        'Wedding / Bridal Dress':['Wedding Ceremony','Engagement','Bridal Shower','Reception'],
        'Activewear / Sportswear':['Gym','Running','Yoga','Sports / Outdoor']
    }

    SEASONS_DISPLAY = {
        'Casual Dress':'Summer / Spring','Formal / Evening Dress':'All Seasons',
        'Bodycon Dress':'Summer / Autumn','Ethnic Wear (Saree/Kurti/Lehenga)':'All Seasons',
        'Maxi Dress':'Summer / Spring','Shirt':'All Seasons',
        'Trousers / Formal Pants':'All Seasons','Traditional / Festival Outfit':'All Seasons',
        'Skirt':'Summer / Spring','Jacket / Blazer':'Autumn / Winter',
        'T-Shirt / Casual Top':'Summer / Spring','Jumpsuit / Romper':'Summer / Spring',
        'Jeans / Denim':'All Seasons','Sweater / Hoodie / Knitwear':'Winter / Autumn',
        'Coat / Overcoat / Puffer':'Winter','Shorts':'Summer',
        'Swimwear / Beachwear':'Summer','Saree (Indian)':'All Seasons',
        'Wedding / Bridal Dress':'All Seasons','Activewear / Sportswear':'All Seasons'
    }

    RATINGS = {
        'Casual Dress':8,'Formal / Evening Dress':10,'Bodycon Dress':9,
        'Ethnic Wear (Saree/Kurti/Lehenga)':10,'Maxi Dress':8,'Shirt':7,
        'Trousers / Formal Pants':7,'Traditional / Festival Outfit':10,
        'Skirt':8,'Jacket / Blazer':9,'T-Shirt / Casual Top':6,
        'Jumpsuit / Romper':8,'Jeans / Denim':8,'Sweater / Hoodie / Knitwear':7,
        'Coat / Overcoat / Puffer':8,'Shorts':6,'Swimwear / Beachwear':7,
        'Saree (Indian)':10,'Wedding / Bridal Dress':10,'Activewear / Sportswear':7
    }

    SHOP_LINKS = {
        'Casual Dress':'https://www.myntra.com/casual-dresses',
        'Formal / Evening Dress':'https://www.myntra.com/formal-dresses',
        'Bodycon Dress':'https://www.myntra.com/bodycon-dresses',
        'Ethnic Wear (Saree/Kurti/Lehenga)':'https://www.myntra.com/ethnic-wear',
        'Maxi Dress':'https://www.myntra.com/maxi-dresses',
        'Shirt':'https://www.myntra.com/shirts',
        'Trousers / Formal Pants':'https://www.myntra.com/trousers',
        'Traditional / Festival Outfit':'https://www.myntra.com/traditional-wear',
        'Skirt':'https://www.myntra.com/skirts',
        'Jacket / Blazer':'https://www.myntra.com/jackets',
        'T-Shirt / Casual Top':'https://www.myntra.com/tshirts',
        'Jumpsuit / Romper':'https://www.myntra.com/jumpsuits',
        'Jeans / Denim':'https://www.myntra.com/jeans',
        'Sweater / Hoodie / Knitwear':'https://www.myntra.com/sweatshirts',
        'Coat / Overcoat / Puffer':'https://www.myntra.com/coats',
        'Shorts':'https://www.myntra.com/shorts',
        'Swimwear / Beachwear':'https://www.myntra.com/swimwear',
        'Saree (Indian)':'https://www.myntra.com/sarees',
        'Wedding / Bridal Dress':'https://www.myntra.com/bridal-wear',
        'Activewear / Sportswear':'https://www.myntra.com/activewear'
    }

    HISTORY_CSV = '/content/dress_history.csv'
    CSV_COLUMNS = ['No', 'Date', 'Time', 'Class', 'Confidence (%)', 'Season', 'Rating (/10)']

    def strip_emoji(text):
        return re.sub(r'[^\x00-\x7F]+', '', str(text)).strip()


    def _csv_to_history(path):
        """Read CSV and convert rows back to history-dict format."""
        records = []
        try:
            with open(path, newline='', encoding='utf-8-sig') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    date = row.get('Date', '').strip()
                    time = row.get('Time', '').strip()
                    records.append({
                        'time':       f"{date} {time}".strip(),
                        'label':      row.get('Class', '').strip(),
                        'confidence': float(row.get('Confidence (%)', 0) or 0),
                        'season':     row.get('Season', '').strip(),
                        'rating':     int(row.get('Rating (/10)', 0) or 0),
                    })
        except Exception as e:
            print(f"[History] Could not read CSV: {e}")
        return records


    def load_history():
        """
        Load history from CSV if it exists, otherwise return empty list.
        This is called on every access so the CSV is always the source of truth.
        """
        if os.path.exists(HISTORY_CSV):
            return _csv_to_history(HISTORY_CSV)
        return []


    def save_history(entry):
        """
        Append a new entry to the CSV immediately.
        Creates the file with headers if it doesn't exist yet.
        """
        file_exists = os.path.exists(HISTORY_CSV)
        existing    = load_history()          # read current records

        
        existing.insert(0, entry)
        existing = existing[:50]              # keep max 50 records

        
        try:
            with open(HISTORY_CSV, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                writer.writerow(CSV_COLUMNS)
                for i, h in enumerate(reversed(existing), 1):   # oldest → newest
                    parts = h.get('time', ' ').split(' ')
                    writer.writerow([
                        i,
                        parts[0] if len(parts) > 0 else '',
                        parts[1] if len(parts) > 1 else '',
                        strip_emoji(h.get('label', '')),
                        h.get('confidence', ''),
                        strip_emoji(h.get('season', '')),
                        h.get('rating', '')
                    ])
            print(f"[History] Saved — {len(existing)} records in {HISTORY_CSV}")
        except Exception as e:
            print(f"[History] Save error: {e}")


    def export_csv():
        """
        Export button handler.
        CSV is already kept up-to-date by save_history(), so just return the path.
        If no history exists yet, inform the user.
        """
        if not os.path.exists(HISTORY_CSV) or os.path.getsize(HISTORY_CSV) == 0:
            print("[Export] No history to export yet.")
            return None
        print(f"[Export] Returning {HISTORY_CSV} for download.")
        return HISTORY_CSV          # Gradio gr.File will serve this for download
                                    # Excel opens UTF-8-sig CSV correctly on double-click


    # ─────────────────────────────────────────────────────────────────────────────
    # ── CHARTS ───────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────

    def make_bar_chart(probs, labels):
        indices   = np.argsort(probs)[::-1][:6]
        top_probs = [probs[i]*100 for i in indices]
        top_lbls  = [labels[i]    for i in indices]
        fig, ax   = plt.subplots(figsize=(7, 3.5))
        colors    = ['#C084FC','#A855F7','#9333EA','#7E22CE','#6B21A8','#581C87']
        bars = ax.barh(top_lbls[::-1], top_probs[::-1],
                    color=colors[::-1], edgecolor='white', height=0.55)
        for bar, val in zip(bars, top_probs[::-1]):
            ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9, fontweight='bold', color='#1e1e2e')
        ax.set_xlim(0, 115)
        ax.set_xlabel('Confidence %', fontsize=9)
        ax.set_title('Top 6 Predictions', fontsize=11, fontweight='bold', color='#1e1e2e')
        ax.spines[['top','right','left']].set_visible(False)
        ax.tick_params(axis='y', labelsize=8)
        fig.patch.set_facecolor('#f8f4ff'); ax.set_facecolor('#f8f4ff')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
        plt.close(); buf.seek(0)
        return Image.open(buf).copy()


    def make_stats_pie():
        history = load_history()
        if not history: return None
        counts  = Counter([h['label'] for h in history])
        labels  = list(counts.keys())
        sizes   = list(counts.values())
        colors  = plt.cm.Set3(np.linspace(0, 1, len(labels)))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.pie(sizes, labels=labels, autopct='%1.0f%%', colors=colors,
            startangle=140, textprops={'fontsize': 7})
        ax.set_title(f'Classification Stats  ({len(history)} total)', fontweight='bold')
        fig.patch.set_facecolor('#f8f4ff')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
        plt.close(); buf.seek(0)
        return Image.open(buf).copy()


    def make_stats_bar():
        history = load_history()
        if not history: return None
        counts  = Counter([h['label'] for h in history])
        labels  = list(counts.keys())
        sizes   = list(counts.values())
        colors  = plt.cm.Purples(np.linspace(0.4, 0.9, len(labels)))
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(range(len(labels)), sizes, color=colors, edgecolor='white')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)
        ax.set_ylabel('Count', fontsize=9)
        ax.set_title('Predictions per Class', fontsize=11, fontweight='bold')
        for bar, val in zip(bars, sizes):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                    str(val), ha='center', fontsize=8, fontweight='bold')
        ax.spines[['top','right']].set_visible(False)
        fig.patch.set_facecolor('#f8f4ff'); ax.set_facecolor('#f8f4ff')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
        plt.close(); buf.seek(0)
        return Image.open(buf).copy()


    def make_confidence_trend():
        history = load_history()
        if len(history) < 2: return None
        recent = history[:20][::-1]
        confs  = [h.get('confidence', 0) for h in recent]
        times  = [h.get('time','') for h in recent]
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(range(len(confs)), confs, 'o-', color='#A855F7', linewidth=2, markersize=5)
        ax.fill_between(range(len(confs)), confs, alpha=0.15, color='#C084FC')
        ax.set_xticks(range(len(times)))
        ax.set_xticklabels(times, rotation=40, ha='right', fontsize=6)
        ax.set_ylabel('Confidence %', fontsize=9)
        ax.set_ylim(0, 105)
        ax.set_title('Confidence Trend (Last 20)', fontsize=11, fontweight='bold')
        ax.spines[['top','right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        fig.patch.set_facecolor('#f8f4ff'); ax.set_facecolor('#f8f4ff')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
        plt.close(); buf.seek(0)
        return Image.open(buf).copy()


    # ── History Table ─────────────────────────────────────────────────────────────
    def get_history_table():
        history = load_history()
        if not history:
            return '*No predictions yet. Classify an image to see history.*'
        rows  = '| # | Time | Class | Confidence | Season | Rating |\n'
        rows += '|---|------|-------|-----------|--------|--------|\n'
        for i, h in enumerate(history[:10], 1):
            rows += (f"| {i} | {h.get('time','')} | {h.get('label','')} | "
                    f"{h.get('confidence','')}% | {h.get('season','')} | "
                    f"{h.get('rating','')}/10 |\n")
        return rows


    def refresh_history():
        return get_history_table(), make_stats_pie(), make_stats_bar(), make_confidence_trend()


    # ── Clear (UI only, history CSV untouched) ────────────────────────────────────
    def clear_all():
        return None, {}, "", None


    # ── Classify ──────────────────────────────────────────────────────────────────
    def classify(image):
        if image is None:
            return {}, 'Please upload an image or use the camera.', None

        inputs = processor(
            text=CLASS_DESCRIPTIONS,
            images=image,
            return_tensors='pt',
            padding=True
        )
        with torch.no_grad():
            probs = model(**inputs).logits_per_image.softmax(dim=1).squeeze().numpy()

        result    = {CLASS_LABELS[i]: float(probs[i]) for i in range(len(CLASS_LABELS))}
        top_idx   = int(np.argmax(probs))
        top_label = CLASS_LABELS[top_idx]
        top_conf  = probs[top_idx] * 100
        emoji     = CLASS_EMOJI.get(top_label, '👗')

        season  = SEASONS_DISPLAY.get(top_label, 'All Seasons')
        rating  = RATINGS.get(top_label, 7)
        stars   = '⭐' * rating + '☆' * (10 - rating)
        tip     = STYLE_TIPS.get(top_label, '')
        occ_str = ' · '.join(OCCASIONS.get(top_label, []))
        shop    = SHOP_LINKS.get(top_label, 'https://www.myntra.com')

        # ✅ Save to CSV immediately — persists across sessions
        save_history({
            'time':       datetime.now().strftime('%d/%m %H:%M'),
            'label':      top_label,
            'confidence': round(float(top_conf), 1),
            'season':     season,
            'rating':     rating
        })

        info = f"""## {emoji} {top_label}

    | Detail | Value |
    |--------|-------|
    | Confidence | **{top_conf:.1f}%** |
    | Outfit Rating | {stars} ({rating}/10) |
    | Best Season | {season} |
    | Shop Now | [Click here — Myntra]({shop}) |

    **Perfect Occasions:** {occ_str}

    **Style Tip:** {tip}
    """
        bar = make_bar_chart(probs, CLASS_LABELS)
        return result, info, bar


    # ── UI ────────────────────────────────────────────────────────────────────────
    css = "footer { display: none !important; }"

    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="purple", secondary_hue="violet", neutral_hue="slate",
            font=gr.themes.GoogleFont("Playfair Display")
        ),
        css=css, title="DressAI"
    ) as app:

        gr.Markdown("""
        # 👗 DressAI — Smart Fashion Classifier
        *Powered by CLIP (OpenAI) · No Training · Instant Results*
        **20 Classes** · Ethnic Wear · Saree · Bridal · Activewear · 3 Charts · History
        """)

        with gr.Tabs():

            # ── Tab 1: Classify ──────────────────────────────────────────────
            with gr.TabItem("📸 Classify"):
                with gr.Row():
                    with gr.Column(scale=1):
                        image_input = gr.Image(
                            type='pil', label='Upload Image or Use Camera',
                            sources=['upload','webcam','clipboard'], height=300
                        )
                        with gr.Row():
                            classify_btn = gr.Button("🔍 Classify", variant="primary")
                            clear_btn    = gr.Button("🗑️ Clear",    variant="secondary")
                        gr.Markdown("*Supports JPG · PNG · WEBP · Webcam*")

                    with gr.Column(scale=2):
                        label_output = gr.Label(num_top_classes=6, label='Top Predictions')
                        info_output  = gr.Markdown()

                gr.Markdown("### 📊 Charts")
                with gr.Row():
                    bar_out = gr.Image(label='Confidence Bar Chart', height=280)

            # ── Tab 2: History ───────────────────────────────────────────────
            with gr.TabItem("🕐 History"):
                gr.Markdown("""
    ### Last 10 Predictions
                """)
                history_md = gr.Markdown()
                with gr.Row():
                    refresh_btn = gr.Button("🔄 Refresh",    variant="secondary")
                    export_btn  = gr.Button("📥 Export CSV ", variant="primary")
                csv_file = gr.File(label="⬇️ Download — double-click to open in Excel")
                gr.Markdown("### 📈 Analytics Charts")
                with gr.Row():
                    pie_out  = gr.Image(label='Category Pie Chart',    height=320)
                    bar2_out = gr.Image(label='Predictions Bar Chart', height=320)
                trend_out = gr.Image(label='Confidence Trend',         height=260)

                refresh_btn.click(fn=refresh_history,
                                outputs=[history_md, pie_out, bar2_out, trend_out])
                export_btn.click(fn=export_csv, outputs=csv_file)

            # ── Tab 3: About ─────────────────────────────────────────────────
            with gr.TabItem("ℹ️ About"):
                gr.Markdown("""
    ## About DressAI
    **Model:** CLIP by OpenAI — trained on 400 million image-text pairs. Zero-shot, no training needed.

    ## 20 Supported Classes
    | # | Class | Examples |
    |---|-------|---------|
    | 1 | Casual Dress | Sundress, day dress |
    | 2 | Formal / Evening Dress | Gown, cocktail dress |
    | 3 | Bodycon Dress | Tight fitted, figure hugging |
    | 4 | Ethnic Wear | Kurti, Salwar, Lehenga |
    | 5 | Maxi Dress | Long, flowy, floor-length |
    | 6 | Shirt | Formal, casual, oxford |
    | 7 | Trousers / Formal Pants | Chinos, slacks |
    | 8 | Traditional / Festival | Regional, cultural wear |
    | 9 | Skirt | Mini, pencil, A-line |
    | 10 | Jacket / Blazer | Suit jacket, sport coat |
    | 11 | T-Shirt / Casual Top | Crop top, tank top |
    | 12 | Jumpsuit / Romper | Dungaree, overalls |
    | 13 | Jeans / Denim | Skinny, wide-leg, bootcut |
    | 14 | Sweater / Hoodie | Knitwear, pullover |
    | 15 | Coat / Overcoat | Trench, puffer, winter coat |
    | 16 | Shorts | Bermuda, denim shorts |
    | 17 | Swimwear / Beachwear | Bikini, one-piece |
    | 18 | Saree (Indian) | Silk, cotton, designer saree |
    | 19 | Wedding / Bridal Dress | Bridal gown, white dress |
    | 20 | Activewear / Sportswear | Gym wear, leggings, yoga |
                """)

        # ── Event Handlers ────────────────────────────────────────────────────
        classify_outputs = [label_output, info_output, bar_out]

        classify_btn.click(fn=classify, inputs=image_input, outputs=classify_outputs)

        clear_btn.click(
            fn=clear_all,
            outputs=[image_input, label_output, info_output, bar_out]
        )

    app.launch(share=True, show_error=True)
    print('\nOpen the gradio.live link above!')

Model already loaded!


/tmp/ipykernel_1308/167055844.py:435: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7cfd24ee7c1b202359.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Open the gradio.live link above!
